# GA Grid Search Results Analysis

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import plotly.graph_objs as go

In [3]:
import pandas as pd
import ast
import csv

def load_ga_results(filepath):
    """
    Load GA results CSV and parse the lists back to Python lists.
    Each cell is converted from string to list of floats.
    """
    df = pd.read_csv(filepath, quoting=csv.QUOTE_NONNUMERIC)

    # Convert string representations of lists to actual lists
    for col in df.columns:
        df[col] = df[col].apply(ast.literal_eval)

    return df


In [9]:
df.head(1)

,blockwise|global_perm|elitism_True,blockwise|global_perm|elitism_False,blockwise|random_swap|elitism_True,blockwise|random_swap|elitism_False,blockwise|between_teams|elitism_True,blockwise|between_teams|elitism_False,positionbased|global_perm|elitism_True,positionbased|global_perm|elitism_False,positionbased|random_swap|elitism_True,positionbased|random_swap|elitism_False,...,blockwise_2child|random_swap|elitism_True,blockwise_2child|random_swap|elitism_False,blockwise_2child|between_teams|elitism_True,blockwise_2child|between_teams|elitism_False,positionbased_2child|global_perm|elitism_True,positionbased_2child|global_perm|elitism_False,positionbased_2child|random_swap|elitism_True,positionbased_2child|random_swap|elitism_False,positionbased_2child|between_teams|elitism_True,positionbased_2child|between_teams|elitism_False
0,"[0.7662486599270054, 0.9034203067406401, 0.720...","[0.813953488372094, 0.8406742291914676, 0.8139...","[0.7382459592813446, 0.8264761545753786, 0.710...","[0.7206886658801452, 0.695865742757577, 0.7514...","[0.8027094836259836, 0.7153311448094616, 0.877...","[0.7382459592813441, 0.7446808510638293, 0.738...","[0.8027094836259859, 0.710186101862879, 0.7153...","[0.751462610817912, 0.813953488372094, 0.85719...","[0.8264761545753786, 0.7382459592813456, 0.700...","[0.8027094836259859, 0.8027094836259859, 0.877...",...,"[0.7830790539176476, 0.8571928494377958, 0.840...","[0.7743698787230197, 0.9034203067406401, 0.783...","[0.7004674983532959, 0.7586344183486812, 0.813...","[0.8571928494377958, 0.7743698787230208, 0.720...","[0.7206886658801452, 0.7446808510638293, 0.751...","[0.7153311448094631, 0.7662486599270085, 0.751...","[0.7052365387324704, 0.8406742291914676, 0.726...","[0.7206886658801457, 0.7830790539176461, 0.751...","[0.8027094836259859, 0.7662486599270054, 0.732...","[0.8027094836259836, 0.682952457518711, 0.7153..."


## Summary of Final Fitness per Configuration

In [7]:
final_gen_fitness = df.tail(1).T
final_gen_fitness.columns = ['Final Fitness']
final_gen_fitness = final_gen_fitness.sort_values('Final Fitness', ascending=False)
display(final_gen_fitness)

,Final Fitness
blockwise|global_perm|elitism_True,"[0.9459459459459473, 0.9459459459459473, 0.945..."
positionbased|random_swap|elitism_False,"[0.9459459459459473, 0.9459459459459473, 0.945..."
blockwise_2child|between_teams|elitism_False,"[0.9459459459459473, 0.9459459459459473, 0.945..."
blockwise_2child|between_teams|elitism_True,"[0.9459459459459473, 0.9459459459459473, 0.945..."
blockwise_2child|random_swap|elitism_False,"[0.9459459459459473, 0.9459459459459473, 0.945..."
blockwise_2child|random_swap|elitism_True,"[0.9459459459459473, 0.9459459459459473, 0.945..."
blockwise_2child|global_perm|elitism_False,"[0.9459459459459473, 0.9459459459459473, 0.945..."
blockwise|global_perm|elitism_False,"[0.9459459459459473, 0.9459459459459473, 0.945..."
blockwise_2child|global_perm|elitism_True,"[0.9459459459459473, 0.9459459459459473, 0.945..."
blockwise|between_teams|elitism_False,"[0.9459459459459473, 0.9459459459459473, 0.945..."


## Plot Fitness Progression for Top 10 Configurations

by early convergence

In [15]:

def find_first_max_position(series):
    max_value = series.max()
    first_pos = series[series == max_value].index[0]
    return first_pos

# calculate the moment where reaches the maximum
first_max_positions = {}
for col in df.columns:
    first_max_positions[col] = find_first_max_position(df[col])

# order by the earliest
sorted_first_max = sorted(first_max_positions.items(), key=lambda x: x[1])


top10_early_labels = [label for label, _ in sorted_first_max[:10]]


fig = go.Figure()
for label in top10_early_labels:
    gen_reached = first_max_positions[label]
    fig.add_trace(go.Scatter(
        y=df[label],
        mode='lines',
        name=f"{label} (Gen {gen_reached})"
    ))

fig.update_layout(
    title="Top 10 Configurations - Earliest Maximum Reached",
    xaxis_title="Generation",
    yaxis_title="Fitness",
    width=900,
    height=500
)
fig.show()


In [8]:
import plotly.graph_objects as go
import numpy as np
import pandas as pd
import json

def plot_fitness_trend_interactive(results_df, title="Fitness Evolution", first_gen=0, last_gen=None):
    """
    Plot interactive evolutionary trends with Plotly (median + std shading).

    Args:
        results_df (pd.DataFrame): DataFrame with lists of fitness per generation and configuration.
        title (str): Title of the plot.
        first_gen (int): Starting generation.
        last_gen (int): Ending generation. If None, uses all generations.
    """
    if last_gen is None:
        last_gen = len(results_df)

    df_slice = results_df.iloc[first_gen:last_gen].copy()

    fig = go.Figure()

    for config in df_slice.columns:
        # Deserialize JSON lists if stored as strings
        if isinstance(df_slice[config].iloc[0], str):
            values = df_slice[config].apply(json.loads)
        else:
            values = df_slice[config]

        # Expand to matrix: generations x runs
        expanded = pd.DataFrame(values.tolist(), index=df_slice.index)

        # Calculate median and std
        median_vals = expanded.median(axis=1)
        std_vals = expanded.std(axis=1)

        # Plot median line
        fig.add_trace(go.Scatter(
            x=expanded.index,
            y=median_vals,
            mode='lines',
            name=config
        ))

        # Plot shaded area (median ± std)
        fig.add_trace(go.Scatter(
            x=list(expanded.index) + list(expanded.index[::-1]),
            y=list(median_vals - std_vals) + list((median_vals + std_vals)[::-1]),
            fill='toself',
            fillcolor='rgba(0,100,80,0.1)',  # Same shading for now
            line=dict(color='rgba(255,255,255,0)'),
            hoverinfo="skip",
            showlegend=False
        ))

    fig.update_layout(
        title=title,
        xaxis_title="Generation",
        yaxis_title="Fitness",
        height=500,
        width=1000,
        legend_title="Configurations",
        template="plotly_white"
    )

    fig.show()


plot_fitness_trend_interactive(df, title="Fitness Evolution of All Configurations", first_gen=0, last_gen=100)


## Compare Elitism vs No Elitism

In [16]:
elitism_cols = [col for col in df.columns if "elitism_True" in col]
no_elitism_cols = [col for col in df.columns if "elitism_False" in col]

def plot_elitism_comparison(cols, title):
    fig = go.Figure()
    for col in cols:
        fig.add_trace(go.Scatter(y=df[col], mode='lines', name=col))
    fig.update_layout(title=title, xaxis_title="Generation", yaxis_title="Fitness")
    fig.show()

plot_elitism_comparison(elitism_cols, "Elitism ON - Fitness Progressions")
plot_elitism_comparison(no_elitism_cols, "Elitism OFF - Fitness Progressions")


## Identify Best Configuration Overall

In [17]:
best_config = final_gen_fitness.head(1)
print("Best Overall Configuration:\n")
display(best_config)


Best Overall Configuration:



,Final Fitness
blockwise|global_perm|elitism_True,0.945946


## Plot Fitness Progression for Worst 4 Configurations

In [18]:
worst3_labels = final_gen_fitness.tail(4).index.tolist()

fig = go.Figure()
for label in worst3_labels:
    fig.add_trace(go.Scatter(
        y=df[label],
        mode='lines',
        name=label
    ))

fig.update_layout(
    title="Worst 4 Configurations - Fitness Progression",
    xaxis_title="Generation",
    yaxis_title="Fitness",
    width=900,
    height=500
)
fig.show()

worst_config = final_gen_fitness.tail(1)
print("Worst Overall Configuration:\n")
display(worst_config)

Worst Overall Configuration:



,Final Fitness
positionbased_2child|global_perm|elitism_True,0.890318
